<a href="https://colab.research.google.com/github/indiaboiii/11501ComputerProgramming/blob/main/Exercise20261015415170490.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

習題一：使用random模組，產生出一個介於0到1之間的小數$r$，使用while迴圈，計算出$r$的前二十位二進位表示$r = 0.b_1b_2b_3\cdots {b_{20}}_2$。輸出格式如附圖Exercise20261015.png

In [2]:
import random

# 產生一個介於 0 到 1 之間的小數 r
r = random.random()

# 用於儲存計算過程中的二進位字串
binary_str = ""
temp_r = r
count = 0

# 使用 while 迴圈計算前二十位二進位表示
while count < 20:
  temp_r *= 2
  if temp_r >= 1:
    binary_str += "1"
    temp_r -= 1
  else:
    binary_str += "0"
  count += 1

# 輸出結果，格式對應附圖範例
print(f"假如 r = {r}")
print(f"r $\approx$ 0.{binary_str}$_2$")

假如 r = 0.812896170822674
r $pprox$ 0.11010000000110011111$_2$


Python 的 `int` 型別採用 Arbitrary-Precision Integer（任意精度整數）設計，能夠精確表示任意大小的整數，其實作方式可參考 CPython 的 `PyLongObject`。與 C/C++ 的 `int` 或 `long long` 不同，Python 的整數不受固定 32 位元或64 位元限制，而是會依據數值大小自動配置更多記憶體，因此只要記憶體足夠，便能表示極大的整數。例如：
```python
n = 10**68
print(n)
```
上述程式可正常執行，表示 Python 能夠精確儲存$10^{68}$，即中文數字中的「無量大數」。


可利用下列程式觀察不同大小整數所需的位元數（bit）與記憶體空間：

```python
import sys
n = 10**68
print(f"bit_length = {n.bit_length()}")
print(f"memory = {sys.getsizeof(n)} bytes")
```

並查閱官方文件，了解：

- `bit_length()`
- `sys.getsizeof()`
- `PyLongObject`

三者之間的關係。

習題二：輸入一個正整數 \(n\)，其中$1 \le n \le 10^{68}$，請將 \(n\) 轉換為中文大寫數字表示法。需支援下列數字單位：萬、億、兆、京、垓、秭、穰、溝、澗、正、載、極、恆河沙、阿僧祇、那由他、不可思議、無量大數。比如輸入31415926，輸出參仟壹佰肆拾壹萬伍仟玖佰貳拾陸。

提示：思考利用字典，建立「數字」與「中文大寫數字」的對應關係。

In [6]:
def number_to_big_chinese(n):
  if n == 0:
    return "零"

  # 定義數字與位數單位
  digits = ["零", "壹", "貳", "參", "肆", "伍", "陸", "柒", "捌", "玖"]
  sub_units = ["", "拾", "佰", "仟"]

  # 定義大單位（每 4 位數為一節）
  large_units = [
      "",
      "萬",
      "億",
      "兆",
      "京",
      "垓",
      "秭",
      "穣",
      "溝",
      "澗",
      "正",
      "載",
      "極",
      "恆河沙",
      "阿僧祇",
      "那由他",
      "不可思議",
      "無量大數",
  ]

  s = str(n)
  chunks = []

  # 從右到左每 4 位數切分為一個區塊 (chunk)
  while s:
    chunks.append(s[-4:])
    s = s[:-4]

  res_parts = []
  zero_flag = False  # 用於處理連續的零區塊

  for i, chunk in enumerate(chunks):
    chunk_val = int(chunk)

    if chunk_val == 0:
      zero_flag = True
      continue

    if zero_flag and res_parts:
      res_parts.append("零")
      zero_flag = False

    # 處理當前 4 位數區塊內的轉換
    padded = chunk.zfill(4)
    chunk_str = ""
    has_internal_zero = False

    # 如果低位區塊長度小於 4 且不是最右邊的區塊，代表有隱含的前導零
    if len(chunk) < 4 and i > 0:
      has_internal_zero = True

    for j, ch in enumerate(padded):
      d = int(ch)
      u = sub_units[3 - j]
      if d == 0:
        if j > 0 and int(padded[j - 1]) != 0:
          has_internal_zero = True
      else:
        if has_internal_zero and chunk_str:
          chunk_str += "零"
          has_internal_zero = False
        chunk_str += digits[d] + u

    # 組合該區塊的大單位（如：萬、億等）
    full_chunk_str = chunk_str + large_units[i]
    res_parts.append(full_chunk_str)

  # 因為是從右到左處理，最後需反轉並串接
  return "".join(reversed(res_parts))



n = 32153519
print(f"輸入：{n}")
print(f"輸出：{number_to_big_chinese(n)}")


輸入：32153519
輸出：參仟貳佰壹拾伍萬參仟伍佰壹拾玖


習題三：請勿使用for、range，而是利用iterable、iter()、next()、while、StopIteration來設計九九乘法表。

In [1]:
# 1. 建立乘數與被乘數的可迭代物件（Iterable，這裡用串列代替 range）
i_list = [1, 2, 3, 4, 5, 6, 7, 8, 9]
j_list = [1, 2, 3, 4, 5, 6, 7, 8, 9]

# 2. 取得外層的 iterator
outer_iter = iter(i_list)

try:
  # 3. 使用 while 迴圈配合 next() 取出外層元素
  while True:
    i = next(outer_iter)

    # 每次進入外層時，都要重新為內層建立一個新的 iterator
    inner_iter = iter(j_list)

    row_str = ""  # 用來儲存每一行的輸出結果
    try:
      while True:
        j = next(inner_iter)
        # 進行相乘並格式化字串
        row_str += f"{i}x{j}={i*j:2d}  "
    except StopIteration:
      # 當內層 iterator 抓完 1~9 之後，會觸發 StopIteration，跳出內層迴圈
      pass

    # 印出該行的九九乘法結果
    print(row_str)

except StopIteration:
  # 當外層 iterator 抓完 1~9 之後，會觸發 StopIteration，結束整個程式
  pass

1x1= 1  1x2= 2  1x3= 3  1x4= 4  1x5= 5  1x6= 6  1x7= 7  1x8= 8  1x9= 9  
2x1= 2  2x2= 4  2x3= 6  2x4= 8  2x5=10  2x6=12  2x7=14  2x8=16  2x9=18  
3x1= 3  3x2= 6  3x3= 9  3x4=12  3x5=15  3x6=18  3x7=21  3x8=24  3x9=27  
4x1= 4  4x2= 8  4x3=12  4x4=16  4x5=20  4x6=24  4x7=28  4x8=32  4x9=36  
5x1= 5  5x2=10  5x3=15  5x4=20  5x5=25  5x6=30  5x7=35  5x8=40  5x9=45  
6x1= 6  6x2=12  6x3=18  6x4=24  6x5=30  6x6=36  6x7=42  6x8=48  6x9=54  
7x1= 7  7x2=14  7x3=21  7x4=28  7x5=35  7x6=42  7x7=49  7x8=56  7x9=63  
8x1= 8  8x2=16  8x3=24  8x4=32  8x5=40  8x6=48  8x7=56  8x8=64  8x9=72  
9x1= 9  9x2=18  9x3=27  9x4=36  9x5=45  9x6=54  9x7=63  9x8=72  9x9=81  
